In [1]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

from src.data.load import load_ratings

ratings = load_ratings()

print("Shape:", ratings.shape)
print(ratings.dtypes)
ratings.head()

Shape: (1048575, 4)
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [2]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)

possible_cells = n_users * n_movies
sparsity = 1 - (n_ratings / possible_cells)

print(f"Users: {n_users:,}")
print(f"Movies: {n_movies:,}")
print(f"Ratings: {n_ratings:,}")
print(f"Possible cells (dense matrix): {possible_cells:,}")
print(f"Sparsity: {sparsity:.4%}")

Users: 7,045
Movies: 22,240
Ratings: 1,048,575
Possible cells (dense matrix): 156,680,800
Sparsity: 99.3308%


In [3]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_id_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

index_to_user_id = {idx: user_id for user_id, idx in user_id_to_index.items()}
index_to_movie_id = {idx: movie_id for movie_id, idx in movie_id_to_index.items()}

print("Number of user indices:", len(user_id_to_index))
print("Number of movie indices:", len(movie_id_to_index))

Number of user indices: 7045
Number of movie indices: 22240


In [4]:
from scipy.sparse import csr_matrix
import numpy as np

row_indices = ratings["userId"].map(user_id_to_index).values
col_indices = ratings["movieId"].map(movie_id_to_index).values
rating_values = ratings["rating"].values

user_item_matrix = csr_matrix(
    (rating_values, (row_indices, col_indices)),
    shape=(n_users, n_movies)
)

print("Matrix shape:", user_item_matrix.shape)
print("Matrix dtype:", user_item_matrix.dtype)
print("Non-zero entries:", user_item_matrix.nnz)
print("Matches ratings count:", user_item_matrix.nnz == n_ratings)

Matrix shape: (7045, 22240)
Matrix dtype: float64
Non-zero entries: 1048575
Matches ratings count: True
